## Import libraries

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib as plt
import seaborn as sns
import requests

## Load Data

In [97]:
path = '..\data\chicago_crimes_2001_2025_raw.csv'
df = pd.read_csv(path)

<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\gus7h\AppData\Local\Temp\ipykernel_25940\3285164765.py:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  path = '..\data\chicago_crimes_2001_2025_raw.csv'


# Cleaning process

First, I want to get rid off 2026 data.

In [98]:
# ── Filter to 2001-2025 ───────────────────────────────────────────────────────
df = df[df['year'].between(2001, 2025)]
print(df['year'].value_counts().sort_index())
print("\nTotal records (2001-2025):", len(df))

year
2001    485958
2002    486831
2003    475999
2004    469443
2005    453788
2006    448198
2007    437108
2008    427216
2009    392866
2010    370567
2011    352052
2012    336375
2013    307612
2014    275880
2015    264887
2016    269959
2017    269287
2018    269147
2019    261702
2020    212700
2021    209656
2022    240002
2023    263282
2024    259007
2025    236574
Name: count, dtype: int64

Total records (2001-2025): 8476096


Looking for missing values:

In [99]:
# ── Missing value summary ─────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)

missing_summary

,missing_count,missing_pct
ward,614818,7.25
community_area,613685,7.24
location,94674,1.12
longitude,94674,1.12
latitude,94674,1.12
y_coordinate,94674,1.12
x_coordinate,94674,1.12
location_description,15520,0.18
primary_type,0,0.00
iucr,0,0.00


Checking data types:

In [100]:
# ── Data type checks ──────────────────────────────────────────────────────────
# fbi_code and iucr are strings, which is correct - they are category codes
print(df.dtypes)
df.info()

# Verify arrest and domestic are boolean
print("arrest unique values:", df["arrest"].unique())
print("domestic unique values:", df["domestic"].unique())
#ok good

id                        int64
case_number                 str
date                        str
block                       str
iucr                        str
primary_type                str
description                 str
location_description        str
arrest                     bool
domestic                   bool
beat                      int64
district                float64
ward                    float64
community_area          float64
fbi_code                    str
year                      int64
updated_on                  str
x_coordinate            float64
y_coordinate            float64
latitude                float64
longitude               float64
location                    str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 8476096 entries, 0 to 8476095
Data columns (total 22 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   case_number           str    
 2   date                  str    
 3   

Type conversions:

In [85]:
df['date'].head(20)

0     2001-01-01T00:00:00.000
1     2001-01-01T00:00:00.000
2     2001-01-01T00:00:00.000
3     2001-01-01T00:00:00.000
4     2001-01-01T00:00:00.000
5     2001-01-01T00:00:00.000
6     2001-01-01T00:00:00.000
7     2001-01-01T00:00:00.000
8     2001-01-01T00:00:00.000
9     2001-01-01T00:00:00.000
10    2001-01-01T00:00:00.000
11    2001-01-01T00:00:00.000
12    2001-01-01T00:00:00.000
13    2001-01-01T00:00:00.000
14    2001-01-01T00:00:00.000
15    2001-01-01T00:00:00.000
16    2001-01-01T00:00:00.000
17    2001-01-01T00:00:00.000
18    2001-01-01T00:00:00.000
19    2001-01-01T00:00:00.000
Name: date, dtype: str

In [ ]:
# district, ward, community_area should be integer
# Int64 (capital I) is used because these columns have missing values
# regular int64 cannot store NaN, Int64 can
df["district"]       = pd.to_numeric(df["district"], errors="coerce").astype("Int64")
df["ward"]           = pd.to_numeric(df["ward"], errors="coerce").astype("Int64")
df["community_area"] = pd.to_numeric(df["community_area"], errors="coerce").astype("Int64")

# Recode categorical text columns to save memory
df["primary_type"]        = df["primary_type"].astype("category")
df["description"]         = df["description"].astype("category")
df["location_description"] = df["location_description"].astype("category")

# Verify missing values did not increase after type conversions
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)
missing_summary

Missing data analysis

In [10]:
# ── Missing data analysis ─────────────────────────────────────────────────────

# Total rows with at least one missing value
print("Rows with at least one missing value:", df.isnull().any(axis=1).sum())
print("That is", round(df.isnull().any(axis=1).sum() / len(df) * 100, 2), "% of the data")

# Which columns are driving the missing values
print("\nMissing values per column for incomplete rows:")
print(df[df.isnull().any(axis=1)][df.columns].isnull().sum())

# ── Is missing data concentrated in certain years? ────────────────────────────
df["has_missing"] = df.isnull().any(axis=1)

missing_by_year = df.groupby("year")["has_missing"].agg(["sum", "count"])
missing_by_year["pct"] = (missing_by_year["sum"] / missing_by_year["count"] * 100).round(2)
missing_by_year.columns = ["missing_count", "total_rows", "missing_pct"]
print(missing_by_year)
# Note: 2001 has high missing data - worth considering in analysis

# Which specific columns are missing by year
df.groupby("year")[["latitude", "longitude", "ward", "community_area"]].apply(lambda x: x.isnull().sum())

#double checking for things that may be considered full but just empty text, seems fine
# ── Check for empty strings masquerading as non-null ─────────────────────────
for col in df.select_dtypes(include="object").columns:
    empty_count = (df[col].astype(str).str.strip() == "").sum()
    nan_string  = (df[col].astype(str) == "nan").sum()
    if empty_count > 0 or nan_string > 0:
        print(col, "- empty strings:", empty_count, "| 'nan' strings:", nan_string)

Rows with at least one missing value: 711753
That is 8.4 % of the data

Missing values per column for incomplete rows:
id                           0
case_number                  0
date                         0
block                        0
iucr                         0
primary_type                 0
description                  0
location_description     15520
arrest                       0
domestic                     0
beat                         0
district                    47
ward                    614818
community_area          613685
fbi_code                     0
year                         0
updated_on                   0
x_coordinate             94674
y_coordinate             94674
latitude                 94674
longitude                94674
location                 94674
dtype: int64
      missing_count  total_rows  missing_pct
year                                        
2001         482056      485958        99.20
2002         141782      486831        29.12
2003  

C:\Users\gus7h\AppData\Local\Temp\ipykernel_25940\575364644.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8476096 entries, 0 to 8476095
Data columns (total 23 columns):
 #   Column                Dtype         
---  ------                -----         
 0   id                    int64         
 1   case_number           str           
 2   date                  datetime64[us]
 3   block                 str           
 4   iucr                  str           
 5   primary_type          category      
 6   description           category      
 7   location_description  category      
 8   arrest                bool          
 9   domestic              bool          
 10  beat                  int64         
 11  district              Int64         
 12  ward                  Int64         
 13  community_area        Int64         
 14  fbi_code              str           
 15  year                  int64         
 16  updated_on            datetime64[us]
 17  x_coordinate          float64       
 18  y_coordinate          float64       
 19  latitude   

## Cleaning issues assigned to me

1. Remove 'location' column.

In [ ]:
df = df.drop(columns=['location'])

2. Remove coordinates outside Chicago

In [ ]:
# ── Remove coordinates outside Chicago ───────────────────────────────────────
# Valid Chicago latitude range: 41.6 - 42.1
# Valid Chicago longitude range: -87.9 - -87.5
print(f'Observations with invalid latitude coordinates: ',len(df[~df['latitude'].between(41.6, 42.1)]))
print(f'Obsevations with null values in latitude coordinates: ',len(df[df['latitude'].isnull()]))
print(f'Thus, observations with non-null values yet out-of-range latitude coordinates:',
      len(df[~df['latitude'].between(41.6, 42.1)]) - len(df[df['latitude'].isnull()]))

print(f'Observations with invalid longitude coordinates: ',len(df[~df['longitude'].between(-87.9, -87.5)]))
print(f'Obsevations with null values in longitude coordinates: ',len(df[df['longitude'].isnull()]))
print(f'Thus, observations with non-null values yet out-of-range longitude coordinates:',
      len(df[~df['longitude'].between(-87.9, -87.5)]) - len(df[df['longitude'].isnull()]))

# It would be safe to delete observations with null-values in either latitude or longitude.
# Deleting 149 observations without valid latitude coordinates might be safe, but doing so
# for 27759 observations without valid longitud coordinates might not. Further exploration
# might be required before making a decision.

Observations with invalid latitude coordinates:  94823
Obsevations with null values in latitude coordinates:  94674
Thus, observations with non-null values yet out-of-range latitude coordinates: 149
Observations with invalid longitude coordinates:  122433
Obsevations with null values in longitude coordinates:  94674
Thus, observations with non-null values yet out-of-range longitude coordinates: 27759


In [ ]:
# How many rows have a null values for latitude/longitude coordinate?
df[df['latitude'].isnull()].groupby('year').count()['id']

year
2001     3079
2002    15301
2003     3957
2004     2231
2005     3859
2006     2633
2007     1400
2008     7318
2009     6901
2010      584
2011      748
2012      909
2013     1199
2014     2125
2015     6979
2016     2735
2017     4351
2018     5624
2019     2531
2020     4729
2021     6787
2022     5119
2023     2037
2024     1459
2025       79
Name: id, dtype: int64

3. Split `date` into `date` and `time`

In [ ]:
date_series = []
for date in df['date']:
    date_series.append(date[0:10])

time_series = []
for time in df['date']:
    time_series.append(time[11:19])

df.drop(columns=['date'])
df['date'] = date_series
df['time'] = time_series

df['date'] = pd.to_datetime(df['date'], format = "%Y-%m-%d")
df['time'] = pd.to_datetime(df['time'], format = "%H:%M:%S").dt.time

df[['date', 'time']].tail(10)

0   2001-01-01
1   2001-01-01
2   2001-01-01
3   2001-01-01
4   2001-01-01
Name: date, dtype: datetime64[us]